In [1]:
import copy
import gzip
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    )
from sklearn.model_selection import StratifiedKFold
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

MTB_GENE_REGIONS = {
    "katG": ("NC_000962.3", 2153889, 2156111),
    "fabG1": ("NC_000962.3", 1673280, 1674183),
    "inhA": ("NC_000962.3", 1674202, 1675011),
    "ahpC": ("NC_000962.3", 2726105, 2726950),
    "rpoB": ("NC_000962.3", 759807, 763325),
    "rpoC": ("NC_000962.3", 763370, 767320),
    "embA": ("NC_000962.3", 4243310, 4245330),
    "embB": ("NC_000962.3", 4246514, 4249810),
    "embC": ("NC_000962.3", 4240713, 4243311),
    "pncA": ("NC_000962.3", 2288681, 2289241),
    "gyrA": ("NC_000962.3", 7301, 9818),
    "gyrB": ("NC_000962.3", 5240, 7267),
    "rrs": ("NC_000962.3", 1471846, 1473382),
    "rrl": ("NC_000962.3", 1473382, 1475491),
    "eis": ("NC_000962.3", 2714065, 2715698),
    "gidB": ("NC_000962.3", 4407590, 4408192),
    "tlyA": ("NC_000962.3", 1917960, 1918590),
    "iniB": ("NC_000962.3", 3985480, 3987076),
    "iniA": ("NC_000962.3", 3987102, 3988602),
    "iniC": ("NC_000962.3", 3988727, 3989990),
    "ndh": ("NC_000962.3", 2102479, 2103985),
    "manB": ("NC_000962.3", 4155253, 4156591),
    "rmlD": ("NC_000962.3", 4154020, 4155231),
    }

FIRST_LINE_CANDIDATES = [
    "INH_BINARY_PHENOTYPE",
    "RIF_BINARY_PHENOTYPE",
    "EMB_BINARY_PHENOTYPE",
    "PZA_BINARY_PHENOTYPE",
    ]


def _open_vcf(path):
    return gzip.open(path, "rt") if str(path).endswith(".gz") else open(path, "r")


def _gt_to_af(gt: str) -> float:
    alleles = re.split(r"[/|]", gt)
    try:
        return sum(1 for a in alleles if a not in ("0", ".")) / len(alleles)
    except Exception:
        return 0.0


def _extract_af(info, parts, header):
    match = re.search(r"AF=([0-9.]+)", info)
    if match:
        return float(match.group(1))
    if len(parts) >= 10 and len(header) >= 10:
        fmt = parts[8].split(":") if len(parts) > 8 else []
        gt_data = parts[9].split(":") if len(parts) > 9 else []
        if fmt and gt_data:
            gt_idx = fmt.index("GT") if "GT" in fmt else 0
            return _gt_to_af(gt_data[gt_idx]) if gt_idx < len(gt_data) else 0.0
    return 0.0


def _pos_to_gene(chrom, pos, gene_regions):
    for gene, (g_chrom, start, end) in gene_regions.items():
        if chrom == g_chrom and start <= pos < end:
            return gene
    return None


def parse_single_sample_vcf(
    vcf_path,
    isolate_id,
    gene_regions=None,
    min_qual=20.0,
    min_af=0.75,
    ):
    gene_regions = gene_regions or MTB_GENE_REGIONS
    records = []
    with _open_vcf(vcf_path) as fh:
        header = []
        for line in fh:
            if line.startswith("##"):
                continue
            if line.startswith("#CHROM"):
                header = line.strip().split("\t")
                continue
            parts = line.strip().split("\t")
            if len(parts) < 8:
                continue
            chrom, pos, _, ref, alt, qual, _, info = parts[:8]
            pos = int(pos)

            try:
                if float(qual) < min_qual:
                    continue
            except ValueError:
                pass

            if len(ref) != 1 or len(alt) != 1 or alt == ".":
                continue
            if _extract_af(info, parts, header) < min_af:
                continue

            gene = _pos_to_gene(chrom, pos, gene_regions)
            if gene is None:
                continue
            records.append((isolate_id, f"{gene}_{ref}{pos}{alt}", gene))
    return records


def load_vcf_directory(
    vcf_dir,
    gene_regions=None,
    pattern="*.vcf*",
    min_qual=20.0,
    min_af=0.75,
    ):
    gene_regions = gene_regions or MTB_GENE_REGIONS
    vcfs = sorted(Path(vcf_dir).glob(pattern))
    if not vcfs:
        raise FileNotFoundError(f"No VCFs matching '{pattern}' in {vcf_dir}")

    all_records = []
    for vcf_path in vcfs:
        isolate_id = vcf_path.name.replace(".vcf.gz", "").replace(".vcf", "")
        recs = parse_single_sample_vcf(
            vcf_path,
            isolate_id,
            gene_regions=gene_regions,
            min_qual=min_qual,
            min_af=min_af,
            )
        all_records.extend(recs)

    isolate_snp_df = pd.DataFrame(all_records, columns=["isolate_id", "snp_id", "gene"])
    if isolate_snp_df.empty:
        raise RuntimeError("No SNP records found after VCF parsing and filtering.")

    print(
        f"Loaded SNP table: {isolate_snp_df['isolate_id'].nunique()} isolates, "
        f"{isolate_snp_df['snp_id'].nunique()} unique SNPs"
        )
    return isolate_snp_df


def load_phenotype_table(phenotype_csv, first_line_candidates=None):
    first_line_candidates = first_line_candidates or FIRST_LINE_CANDIDATES

    pheno_columns = pd.read_csv(phenotype_csv, nrows=0).columns.tolist()
    drug_cols = [col for col in first_line_candidates if col in pheno_columns]
    if not drug_cols:
        raise ValueError("No first-line phenotype columns found in phenotype table.")

    phenotype_raw = pd.read_csv(phenotype_csv)
    if "ENA_SAMPLE" not in phenotype_raw.columns:
        raise KeyError("Expected ENA_SAMPLE column in phenotype table.")

    missing = [col for col in drug_cols if col not in phenotype_raw.columns]
    if missing:
        raise KeyError(f"Missing phenotype columns: {missing}")

    phenotype_df = phenotype_raw[["ENA_SAMPLE", *drug_cols]].copy()
    phenotype_df = phenotype_df.rename(columns={"ENA_SAMPLE": "isolate_id"})

    label_map = {"R": 1.0, "S": 0.0}
    for col in drug_cols:
        phenotype_df[col] = (
            phenotype_df[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .map(label_map)
            .astype(float)
            )

    print("Using first-line drugs:", drug_cols)
    return phenotype_df, drug_cols


def compute_pmi_embeddings(isolate_snp_df, embed_dim=64, min_count=5):
    snp_counts = isolate_snp_df["snp_id"].value_counts()
    valid_snps = snp_counts[snp_counts >= min_count].index.tolist()
    df = isolate_snp_df[isolate_snp_df["snp_id"].isin(valid_snps)].copy()

    isolates = df["isolate_id"].unique().tolist()
    snps = df["snp_id"].unique().tolist()
    iso_idx = {sid: i for i, sid in enumerate(isolates)}
    snp_idx = {sid: i for i, sid in enumerate(snps)}

    rows = [iso_idx[r] for r in df["isolate_id"]]
    cols = [snp_idx[r] for r in df["snp_id"]]
    mat = csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(len(isolates), len(snps)),
        dtype=np.float32,
        )

    co = (mat.T @ mat).toarray()
    total = co.sum()
    p_ij = co / total
    p_i = co.sum(axis=1, keepdims=True) / total
    p_j = co.sum(axis=0, keepdims=True) / total

    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log(p_ij / (p_i * p_j + 1e-12))
    pmi = np.nan_to_num(pmi, nan=0.0, neginf=0.0)
    pmi = np.maximum(pmi, 0.0)

    k = max(1, min(embed_dim, min(pmi.shape) - 1))
    u, s, _ = svds(csr_matrix(pmi), k=k)
    embeddings = (u * np.sqrt(s)).astype(np.float32)

    if embeddings.shape[1] < embed_dim:
        pad = np.zeros((embeddings.shape[0], embed_dim - embeddings.shape[1]), dtype=np.float32)
        embeddings = np.concatenate([embeddings, pad], axis=1)

    return embeddings, snps


def prepare_hgat_inputs(
    vcf_dir="./vcf(2024)",
    phenotype_csv="./CRyPTIC_reuse_table_20240917.csv",
    embed_dim=64,
    min_snp_count=5,
    min_qual=20.0,
    min_af=0.75,
    ):
    isolate_snp_df = load_vcf_directory(vcf_dir, min_qual=min_qual, min_af=min_af)
    phenotype_df, drug_cols = load_phenotype_table(phenotype_csv)
    snp_embeddings, snp_list = compute_pmi_embeddings(
        isolate_snp_df,
        embed_dim=embed_dim,
        min_count=min_snp_count,
        )

    print(f"SNP embeddings: {snp_embeddings.shape} (n_snps x embed_dim)")
    return {
        "isolate_snp_df": isolate_snp_df,
        "phenotype_df": phenotype_df,
        "snp_embeddings": snp_embeddings,
        "snp_list": snp_list,
        "DRUG_COLS": drug_cols,
        }

/mnt/bigssd/akshat_home/miniconda3/envs/lma/lib/python3.13/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._show_config()
/mnt/bigssd/akshat_home/miniconda3/envs/lma/lib/python3.13/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Using device: cpu


In [2]:
def build_hetero_graph(
    isolate_snp_df,
    phenotype_df,
    snp_embeddings,
    snp_list,
    drug_cols,
    ):
    snp_emb_map = {snp_id: i for i, snp_id in enumerate(snp_list)}
    df = isolate_snp_df[isolate_snp_df["snp_id"].isin(snp_emb_map)].copy()

    isolates = df["isolate_id"].unique().tolist()
    iso_to_idx = {isolate_id: i for i, isolate_id in enumerate(isolates)}
    genes = sorted(df["gene"].unique().tolist())
    data = HeteroData()

    iso_feats = np.zeros((len(isolates), snp_embeddings.shape[1]), dtype=np.float32)
    for isolate_id in isolates:
        idxs = [
            snp_emb_map[snp_id]
            for snp_id in df[df["isolate_id"] == isolate_id]["snp_id"]
            if snp_id in snp_emb_map
            ]
        if idxs:
            iso_feats[iso_to_idx[isolate_id]] = snp_embeddings[idxs].max(axis=0)

    data["isolate"].x = torch.from_numpy(iso_feats)
    phen = phenotype_df.set_index("isolate_id").reindex(isolates)[drug_cols]
    data["isolate"].y = torch.tensor(phen.values.astype(np.float32))
    data["isolate"].isolate_ids = isolates

    for gene in genes:
        gdf = df[df["gene"] == gene]
        gene_snps = gdf["snp_id"].unique().tolist()
        snp_local = {snp_id: i for i, snp_id in enumerate(gene_snps)}
        emb_idxs = [snp_emb_map[snp_id] for snp_id in gene_snps]

        data[gene].x = torch.from_numpy(snp_embeddings[emb_idxs])
        data[gene].snp_ids = gene_snps

        iso_nodes, snp_nodes = [], []
        for _, row in gdf.iterrows():
            isolate_id = row["isolate_id"]
            snp_id = row["snp_id"]
            if isolate_id in iso_to_idx and snp_id in snp_local:
                iso_nodes.append(iso_to_idx[isolate_id])
                snp_nodes.append(snp_local[snp_id])

        if iso_nodes:
            edge_index = torch.tensor([iso_nodes, snp_nodes], dtype=torch.long)
            data[("isolate", f"has_{gene}", gene)].edge_index = edge_index
            data[(gene, f"in_{gene}", "isolate")].edge_index = edge_index.flip(0)

    n_snp_nodes = sum(data[g].x.size(0) for g in genes if hasattr(data[g], "x"))
    print(
        f"Graph built: {len(isolates)} isolates | {len(genes)} gene types | {n_snp_nodes} SNP nodes"
        )
    return data


class HGATLayer(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        edge_types,
        heads=2,
        dropout=0.3,
        negative_slope=0.2,
    ):
        super().__init__()
        assert out_channels % heads == 0
        self.edge_types = edge_types

        convs = {
            (src, rel, dst): GATConv(
                in_channels=(-1, -1),
                out_channels=out_channels // heads,
                heads=heads,
                dropout=dropout,
                negative_slope=negative_slope,
                add_self_loops=False,
                concat=True,
                )
            for src, rel, dst in edge_types
            }
        self.conv = HeteroConv(convs, aggr="sum")

        self.type_attn_raw = nn.Parameter(torch.zeros(len(edge_types)))
        dst_types = sorted({dst for _, _, dst in edge_types})
        self.norms = nn.ModuleDict({dst: nn.LayerNorm(out_channels) for dst in dst_types})
        self.drop = nn.Dropout(dropout)

    @property
    def type_attn_weights(self):
        return F.softmax(self.type_attn_raw, dim=0)

    def forward(self, x_dict, edge_index_dict):
        raw_out = self.conv(x_dict, edge_index_dict)
        out = {}
        for dst, h in raw_out.items():
            if h is None:
                continue
            h = self.norms[dst](h)
            h = F.elu(h)
            h = self.drop(h)
            out[dst] = h
        return out


def masked_bce_loss(logits, labels, pos_weight=None, l2_lambda=1e-4, params=None):
    mask = ~torch.isnan(labels)
    if mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)

    if pos_weight is not None:
        weight_mat = torch.ones_like(labels, device=logits.device)
        pw = pos_weight.unsqueeze(0).expand_as(labels)
        weight_mat = torch.where(labels == 1.0, pw, weight_mat)
        loss = F.binary_cross_entropy_with_logits(
            logits[mask],
            labels[mask],
            weight=weight_mat[mask],
            reduction="mean",
            )
    else:
        loss = F.binary_cross_entropy_with_logits(logits[mask], labels[mask], reduction="mean")

    if l2_lambda > 0 and params is not None:
        l2 = sum(p.pow(2).sum() for p in params)
        loss = loss + l2_lambda * l2
    return loss


def inductive_label_mask(labels, mask_ratio):
    labels = labels.clone()
    observed = ~torch.isnan(labels)
    n_mask = int(observed.sum().item() * mask_ratio)
    if n_mask == 0:
        return labels
    positions = observed.nonzero(as_tuple=False)
    chosen = positions[torch.randperm(len(positions))[:n_mask]]
    for idx in chosen:
        labels[idx[0], idx[1]] = float("nan")
    return labels


def find_best_thresholds(probs, labels, drug_names):
    thresholds = {}
    grid = np.linspace(0.1, 0.9, 17)
    for i, drug in enumerate(drug_names):
        valid = ~np.isnan(labels[:, i])
        if valid.sum() < 2 or len(np.unique(labels[valid, i])) < 2:
            thresholds[drug] = 0.5
            continue

        yt = labels[valid, i].astype(int)
        yp = probs[valid, i]
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            yb = (yp >= t).astype(int)
            f1 = f1_score(yt, yb, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        thresholds[drug] = best_t
    return thresholds


def compute_metrics(probs, labels, drug_names, thresholds=None):
    results = {}
    for i, drug in enumerate(drug_names):
        valid = ~np.isnan(labels[:, i])
        if valid.sum() < 2 or len(np.unique(labels[valid, i])) < 2:
            continue

        yt = labels[valid, i].astype(int)
        yp = probs[valid, i]
        th = 0.5 if thresholds is None else thresholds.get(drug, 0.5)
        yb = (yp >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yb, labels=[0, 1]).ravel()

        results[drug] = {
            "AUROC": roc_auc_score(yt, yp),
            "AUPRC": average_precision_score(yt, yp),
            "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
            "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
            "F1": f1_score(yt, yb, zero_division=0),
            "Threshold": th,
            }
    return results


@torch.no_grad()
def isolate_attention(model, data, isolate_idx):
    model.eval()
    results = {}
    for layer_idx, layer in enumerate(model.layers):
        for (src, rel, dst), conv in layer.conv.convs.items():
            if src != "isolate" or not rel.startswith("has_"):
                continue

            gene = rel.replace("has_", "")
            edge_index = data.edge_index_dict.get((src, rel, dst))
            if edge_index is None:
                continue

            iso_mask = edge_index[0] == isolate_idx
            if iso_mask.sum() == 0:
                continue

            x_src = data.x_dict.get(src)
            x_dst = data.x_dict.get(dst)
            if x_src is None or x_dst is None:
                continue

            try:
                _, (_, attn) = conv((x_src, x_dst), edge_index, return_attention_weights=True)
            except Exception:
                continue

            attn_iso = attn[iso_mask].mean(dim=1).detach().cpu().numpy()
            snp_idxs = edge_index[1][iso_mask].cpu().numpy()
            snp_ids = getattr(data[dst], "snp_ids", None)
            if snp_ids is None:
                continue

            valid = [int(i) for i in snp_idxs if 0 <= int(i) < len(snp_ids)]
            if not valid:
                continue

            results[f"L{layer_idx+1}_{gene}"] = pd.DataFrame(
                {
                    "snp_id": [snp_ids[i] for i in valid],
                    "attention": attn_iso[: len(valid)],
                    }
                ).sort_values("attention", ascending=False)
    return results

In [3]:
def normalize_drug_name(drug):
    aliases = {
        "INH": "Isoniazid",
        "RIF": "Rifampicin",
        "EMB": "Ethambutol",
        "PZA": "Pyrazinamide",
        }
    return aliases.get(drug, drug)


def extract_drug_token(drug_col):
    return str(drug_col).replace("_BINARY_PHENOTYPE", "")


def _as_pathway_list(value):
    if value is None:
        return []
    if isinstance(value, dict):
        if "pathways" in value:
            return _as_pathway_list(value.get("pathways"))
        if "id" in value:
            return _as_pathway_list(value.get("id"))
        return []
    if isinstance(value, (list, tuple, set)):
        values = value
    else:
        values = [value]
    return [str(v).strip() for v in values if str(v).strip()]


def _derive_pathway_ids(kg):
    pathways = set()

    raw_pathways = kg.get("pathways", {})
    if isinstance(raw_pathways, dict):
        pathways.update(str(k).strip() for k in raw_pathways.keys() if str(k).strip())
    elif isinstance(raw_pathways, (list, tuple, set)):
        pathways.update(str(v).strip() for v in raw_pathways if str(v).strip())

    pathway_to_genes = kg.get("pathway_to_genes", {})
    if isinstance(pathway_to_genes, dict):
        pathways.update(str(k).strip() for k in pathway_to_genes.keys() if str(k).strip())

    gene_to_pathways = kg.get("gene_to_pathways", {})
    if isinstance(gene_to_pathways, dict):
        for vals in gene_to_pathways.values():
            pathways.update(_as_pathway_list(vals))

    drug_to_pathways = kg.get("drug_to_pathways", {})
    if isinstance(drug_to_pathways, dict):
        for vals in drug_to_pathways.values():
            pathways.update(_as_pathway_list(vals))

    return sorted(pathways)


def _build_gene_to_pathway_lookup(kg, valid_pathways=None):
    valid = set(valid_pathways) if valid_pathways is not None else None
    raw = kg.get("gene_to_pathways", {})

    lookup = {}
    if not isinstance(raw, dict):
        return lookup

    for gene, vals in raw.items():
        gene_key = str(gene).strip()
        if not gene_key:
            continue

        pathways = _as_pathway_list(vals)
        if valid is not None:
            pathways = [p for p in pathways if p in valid]
        if not pathways:
            continue

        for key in {gene_key, gene_key.lower(), gene_key.upper()}:
            lookup.setdefault(key, set()).update(pathways)

    return {k: sorted(v) for k, v in lookup.items()}


def _pathways_for_gene(gene, gene_to_pathways):
    gene_key = str(gene).strip()
    return (
        gene_to_pathways.get(gene_key, [])
        or gene_to_pathways.get(gene_key.lower(), [])
        or gene_to_pathways.get(gene_key.upper(), [])
        )


def load_kegg_knowledge_graph(kegg_json_path):
    with open(kegg_json_path, "r") as f:
        kg = json.load(f)

    pathways = _derive_pathway_ids(kg)
    gene_to_pwys = kg.get("gene_to_pathways", {})
    drug_to_pwys = kg.get("drug_to_pathways", {})

    n_gene_links = sum(len(_as_pathway_list(v)) for v in gene_to_pwys.values()) if isinstance(gene_to_pwys, dict) else 0
    n_drug_links = sum(len(_as_pathway_list(v)) for v in drug_to_pwys.values()) if isinstance(drug_to_pwys, dict) else 0

    print(f"Loaded KEGG KG from {kegg_json_path}")
    print(f"Pathway IDs available: {len(pathways)}")
    print(f"Gene->Pathway links: {n_gene_links}")
    print(f"Drug->Pathway links: {n_drug_links}")
    return kg


def build_pathway_aware_hetero_graph(
    isolate_snp_df,
    phenotype_df,
    snp_embeddings,
    snp_list,
    drug_cols,
    kg,
    ):
    data = build_hetero_graph(
        isolate_snp_df=isolate_snp_df,
        phenotype_df=phenotype_df,
        snp_embeddings=snp_embeddings,
        snp_list=snp_list,
        drug_cols=drug_cols,
        )

    pathways = _derive_pathway_ids(kg)
    pwy_to_idx = {p: i for i, p in enumerate(pathways)}
    pwy_feats = np.eye(len(pathways), dtype=np.float32) if pathways else np.zeros((0, 1), dtype=np.float32)
    data["pathway"].x = torch.from_numpy(pwy_feats)
    data["pathway"].pathway_ids = pathways

    snp_gene_types = [nt for nt in data.node_types if nt not in ("isolate", "pathway", "gene")]
    gene_to_pwys = _build_gene_to_pathway_lookup(kg, valid_pathways=pathways)

    connected_genes = [g for g in snp_gene_types if _pathways_for_gene(g, gene_to_pwys)]
    gp_src, gp_dst = [], []
    if connected_genes:
        gene_to_idx = {g: i for i, g in enumerate(connected_genes)}
        for gene in connected_genes:
            for p in _pathways_for_gene(gene, gene_to_pwys):
                if p in pwy_to_idx:
                    gp_src.append(gene_to_idx[gene])
                    gp_dst.append(pwy_to_idx[p])

        if gp_src:
            data["gene"].x = torch.eye(len(connected_genes), dtype=torch.float32)
            data["gene"].gene_ids = connected_genes
            edge_gp = torch.tensor([gp_src, gp_dst], dtype=torch.long)
            data[("gene", "participates_in", "pathway")].edge_index = edge_gp
            data[("pathway", "includes_gene", "gene")].edge_index = edge_gp.flip(0)

    isolate_ids = list(getattr(data["isolate"], "isolate_ids", []))
    iso_to_idx = {iso: i for i, iso in enumerate(isolate_ids)}

    ip_pairs = set()
    for row in isolate_snp_df[["isolate_id", "gene"]].dropna().itertuples(index=False):
        isolate_id, gene = row
        if isolate_id not in iso_to_idx:
            continue
        for pathway in _pathways_for_gene(gene, gene_to_pwys):
            if pathway in pwy_to_idx:
                ip_pairs.add((iso_to_idx[isolate_id], pwy_to_idx[pathway]))

    if ip_pairs:
        iso_nodes = [pair[0] for pair in ip_pairs]
        pwy_nodes = [pair[1] for pair in ip_pairs]
        edge_ip = torch.tensor([iso_nodes, pwy_nodes], dtype=torch.long)
        data[("isolate", "in_pathway", "pathway")].edge_index = edge_ip
        data[("pathway", "has_isolate", "isolate")].edge_index = edge_ip.flip(0)

    print(
        f"Pathway graph: {data['pathway'].x.shape[0]} pathways; "
        f"gene-pathway edges = {len(gp_src)}; "
        f"isolate-pathway edges = {len(ip_pairs)}"
        )
    return data


class PathwayHGAT_AMR(nn.Module):
    def __init__(self, hidden_channels, out_channels, metadata, dropout=0.3):
        super().__init__()
        self.metadata = metadata
        self.hidden_channels = hidden_channels
        self.dropout = dropout

        edge_types = metadata[1]
        self.layers = nn.ModuleList([
            HGATLayer(
                in_channels=-1,
                out_channels=hidden_channels,
                edge_types=edge_types,
                heads=2,
                dropout=dropout,
                ),
            HGATLayer(
                in_channels=hidden_channels,
                out_channels=hidden_channels,
                edge_types=edge_types,
                heads=2,
                dropout=dropout,
                ),
            ])

        self.pathway_ctx = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels),
            )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 3, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_channels),
            )

    def forward(self, x_dict, edge_index_dict):
        h0 = copy.deepcopy(x_dict)
        h1 = self.layers[0](h0, edge_index_dict)
        h2 = self.layers[1](h1, edge_index_dict)

        iso_l1 = h1.get("isolate")
        if iso_l1 is None:
            n_iso = h0["isolate"].size(0)
            iso_l1 = torch.zeros(n_iso, self.hidden_channels, device=h0["isolate"].device)
        iso_l2 = h2.get("isolate", iso_l1)

        if "pathway" in h2 and h2["pathway"].size(0) > 0:
            p = h2["pathway"]
            scores = torch.matmul(iso_l2, p.T) / np.sqrt(max(p.shape[1], 1))
            alpha = torch.softmax(scores, dim=1)
            p_ctx = alpha @ p
            p_ctx = self.pathway_ctx(p_ctx)
        else:
            p_ctx = torch.zeros_like(iso_l2)

        combined = torch.cat([iso_l1, iso_l2, p_ctx], dim=1)
        logits = self.classifier(combined)
        return logits, h2


def build_pathway_model(data, out_channels, hidden_channels=64, dropout=0.3, device=DEVICE):
    model = PathwayHGAT_AMR(
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        metadata=data.metadata(),
        dropout=dropout,
        ).to(device)
    return model


def train_pathway_model(
    data,
    model,
    drug_cols,
    n_splits=5,
    n_epochs=60,
    lr=2e-3,
    weight_decay=1e-4,
    label_mask_ratio=0.15,
    seed=42,
    ):
    y_np = data["isolate"].y.detach().cpu().numpy()
    y_bin = np.nan_to_num(y_np[:, 0], nan=0.0).astype(int)

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(splitter.split(np.zeros(len(y_bin)), y_bin), 1):
        print(f"\nFold {fold}/{n_splits}")
        local_model = copy.deepcopy(model)
        optimizer = torch.optim.Adam(local_model.parameters(), lr=lr, weight_decay=weight_decay)

        train_idx_t = torch.tensor(train_idx, dtype=torch.long, device=DEVICE)
        test_idx_t = torch.tensor(test_idx, dtype=torch.long, device=DEVICE)

        y_train = data["isolate"].y[train_idx_t]
        with torch.no_grad():
            pos_counts = torch.nansum(y_train, dim=0)
            valid_counts = torch.sum(~torch.isnan(y_train), dim=0).float()
            neg_counts = valid_counts - pos_counts
            pos_weight = (neg_counts / (pos_counts + 1e-6)).clamp(min=1.0, max=10.0)

        best_probs, best_labels = None, None
        best_mean_auc = -1.0

        for epoch in range(1, n_epochs + 1):
            local_model.train()
            optimizer.zero_grad()
            logits, _ = local_model(data.x_dict, data.edge_index_dict)

            y_train_masked = inductive_label_mask(
                data["isolate"].y[train_idx_t],
                mask_ratio=label_mask_ratio,
                )
            loss = masked_bce_loss(
                logits[train_idx_t],
                y_train_masked,
                pos_weight=pos_weight,
                params=local_model.parameters(),
                )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), max_norm=1.0)
            optimizer.step()

            if epoch % 10 == 0 or epoch == n_epochs:
                local_model.eval()
                with torch.no_grad():
                    eval_logits, _ = local_model(data.x_dict, data.edge_index_dict)
                    probs = torch.sigmoid(eval_logits[test_idx_t]).detach().cpu().numpy()
                    labels = data["isolate"].y[test_idx_t].detach().cpu().numpy()

                metric_rows = compute_metrics(probs, labels, drug_cols)
                if metric_rows:
                    mean_auc = float(np.mean([m["AUROC"] for m in metric_rows.values()]))
                    if mean_auc > best_mean_auc:
                        best_mean_auc = mean_auc
                        best_probs, best_labels = probs, labels

        if best_probs is None:
            local_model.eval()
            with torch.no_grad():
                eval_logits, _ = local_model(data.x_dict, data.edge_index_dict)
                best_probs = torch.sigmoid(eval_logits[test_idx_t]).detach().cpu().numpy()
                best_labels = data["isolate"].y[test_idx_t].detach().cpu().numpy()

        fold_thresholds = find_best_thresholds(best_probs, best_labels, drug_cols)
        fold_result = compute_metrics(best_probs, best_labels, drug_cols, thresholds=fold_thresholds)
        fold_metrics.append(fold_result)

    summary = {}
    for drug in drug_cols:
        rows = [fm[drug] for fm in fold_metrics if drug in fm]
        if not rows:
            continue
        summary[drug] = {
            metric: (float(np.mean([r[metric] for r in rows])), float(np.std([r[metric] for r in rows])))
            for metric in rows[0].keys()
            }
    return summary


def full_attribution_chain(
    model,
    data,
    isolate_id,
    kegg_kg=None,
    top_k_snps=25,
    ):
    isolate_ids = list(getattr(data["isolate"], "isolate_ids", []))
    if isolate_id not in isolate_ids:
        raise ValueError(f"isolate_id={isolate_id} not found in graph isolate_ids")

    idx = isolate_ids.index(isolate_id)
    snp_attn = isolate_attention(model, data, idx)

    snp_scores = []
    for key, df in snp_attn.items():
        if df.empty:
            continue
        gene = key.split("_", 1)[1] if "_" in key else key
        tmp = df[["snp_id", "attention"]].copy()
        tmp["gene"] = gene
        snp_scores.append(tmp)

    if snp_scores:
        snp_df = pd.concat(snp_scores, ignore_index=True)
        snp_df = snp_df.sort_values("attention", ascending=False).head(top_k_snps).reset_index(drop=True)
    else:
        snp_df = pd.DataFrame(columns=["snp_id", "attention", "gene"])

    gene_df = pd.DataFrame(columns=["gene", "score"])
    if not snp_df.empty:
        gene_df = (
            snp_df.groupby("gene", as_index=False)["attention"]
            .sum()
            .rename(columns={"attention": "score"})
            .sort_values("score", ascending=False)
            .reset_index(drop=True)
            )

    pathway_df = pd.DataFrame(columns=["pathway", "score"])
    if kegg_kg is not None and not gene_df.empty:
        gene_to_pathways = _build_gene_to_pathway_lookup(kegg_kg)
        pathway_scores = {}
        for _, row in gene_df.iterrows():
            gene = row["gene"]
            g_score = float(row["score"])
            for pathway in _pathways_for_gene(gene, gene_to_pathways):
                pathway_scores[pathway] = pathway_scores.get(pathway, 0.0) + g_score
        if pathway_scores:
            pathway_df = (
                pd.DataFrame({"pathway": list(pathway_scores.keys()), "score": list(pathway_scores.values())})
                .sort_values("score", ascending=False)
                .reset_index(drop=True)
                )

    return {
        "snp_level": snp_df,
        "gene_level": gene_df,
        "pathway_level": pathway_df,
        }


def visualize_attribution_chain(chain, top_n=15):
    print("\n=== SNP Level (Top) ===")
    print(chain["snp_level"].head(top_n))
    print("\n=== Gene Level (Top) ===")
    print(chain["gene_level"].head(top_n))
    print("\n=== Pathway Level (Top) ===")
    print(chain["pathway_level"].head(top_n))

In [4]:
def validate_against_known_mechanisms(chain_results, drug_name):
    known_targets = {
        "INH": {"katG", "inhA", "ahpC", "fabG1", "ndh"},
        "RIF": {"rpoB", "rpoC"},
        "EMB": {"embA", "embB", "embC", "iniA", "iniB", "iniC"},
        "PZA": {"pncA", "rpsA"},
        }

    token = extract_drug_token(drug_name)
    targets = known_targets.get(token, set())
    if not targets:
        return {"drug": drug_name, "known_targets": [], "top_genes": [], "overlap": []}

    top_genes = chain_results["gene_level"].head(20)["gene"].tolist()
    overlap = sorted(set(top_genes).intersection(targets))

    return {
        "drug": drug_name,
        "known_targets": sorted(targets),
        "top_genes": top_genes,
        "overlap": overlap,
        "hit_rate": len(overlap) / max(1, len(targets)),
        }


def summarize_validation(validation_results):
    rows = []
    for drug, result in validation_results.items():
        rows.append(
            {
                "drug": drug,
                "hit_rate": result.get("hit_rate", 0.0),
                "overlap": ", ".join(result.get("overlap", [])),
                "n_overlap": len(result.get("overlap", [])),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["drug", "hit_rate", "overlap", "n_overlap"])
    return pd.DataFrame(rows).sort_values(["hit_rate", "n_overlap"], ascending=False).reset_index(drop=True)


def run_pathway_aware_pipeline(
    isolate_snp_df=None,
    phenotype_df=None,
    snp_embeddings=None,
    snp_list=None,
    drug_cols=None,
    kegg_json_path="./kegg_data/tb_drug_knowledge_graph.json",
    known_isolate_id=None,
    auto_prepare_inputs=True,
    vcf_dir="./vcf(2024)",
    phenotype_csv="./CRyPTIC_reuse_table_20240917.csv",
    embed_dim=64,
    min_snp_count=5,
    min_qual=20.0,
    min_af=0.75,
    train_config=None,
    ):
    if auto_prepare_inputs and any(
        x is None for x in [isolate_snp_df, phenotype_df, snp_embeddings, snp_list, drug_cols]
        ):
        prepared = prepare_hgat_inputs(
            vcf_dir=vcf_dir,
            phenotype_csv=phenotype_csv,
            embed_dim=embed_dim,
            min_snp_count=min_snp_count,
            min_qual=min_qual,
            min_af=min_af,
            )
        isolate_snp_df = prepared["isolate_snp_df"]
        phenotype_df = prepared["phenotype_df"]
        snp_embeddings = prepared["snp_embeddings"]
        snp_list = prepared["snp_list"]
        drug_cols = prepared["DRUG_COLS"]

    kg = load_kegg_knowledge_graph(kegg_json_path)
    data = build_pathway_aware_hetero_graph(
        isolate_snp_df=isolate_snp_df,
        phenotype_df=phenotype_df,
        snp_embeddings=snp_embeddings,
        snp_list=snp_list,
        drug_cols=drug_cols,
        kg=kg,
        )
    data = data.to(DEVICE)

    model = build_pathway_model(
        data=data,
        out_channels=len(drug_cols),
        hidden_channels=64,
        dropout=0.3,
        device=DEVICE,
        )

    cfg = {
        "n_splits": 5,
        "n_epochs": 60,
        "lr": 2e-3,
        "weight_decay": 1e-4,
        "label_mask_ratio": 0.15,
        "seed": 42,
        }
    if train_config:
        cfg.update(train_config)

    metrics = train_pathway_model(
        data=data,
        model=model,
        drug_cols=drug_cols,
        n_splits=cfg["n_splits"],
        n_epochs=cfg["n_epochs"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        label_mask_ratio=cfg["label_mask_ratio"],
        seed=cfg["seed"],
        )

    if known_isolate_id is None:
        known_isolate_id = data["isolate"].isolate_ids[0]

    chain = full_attribution_chain(
        model=model,
        data=data,
        isolate_id=known_isolate_id,
        kegg_kg=kg,
        top_k_snps=25,
        )

    validations = {
        drug: validate_against_known_mechanisms(chain, drug)
        for drug in drug_cols
        }

    return {
        "metrics": metrics,
        "attribution_chain": chain,
        "biological_validation": validations,
        "drug_cols": drug_cols,
        "known_isolate_id": known_isolate_id,
        "data": data,
        "model": model,
        "kg": kg,
        }

In [5]:
# KEGG pathway-only setup: RIF single-head classification
FIRST_LINE_CANDIDATES = ["RIF_BINARY_PHENOTYPE"]
print("KEGG single-head mode:", FIRST_LINE_CANDIDATES[0])

KEGG single-head mode: RIF_BINARY_PHENOTYPE


In [6]:
# Step 1: Data import + preprocessing (feature engineering)
prepared = prepare_hgat_inputs(
    vcf_dir="./vcf(2024)",
    phenotype_csv="./CRyPTIC_reuse_table_20240917.csv",
    embed_dim=64,
    min_snp_count=5,
    min_qual=20.0,
    min_af=0.75,
    )

isolate_snp_df = prepared["isolate_snp_df"]
phenotype_df = prepared["phenotype_df"]
snp_embeddings = prepared["snp_embeddings"]
snp_list = prepared["snp_list"]
DRUG_COLS = prepared["DRUG_COLS"]

print("\nPrepared inputs:")
print(f"  Isolates: {isolate_snp_df['isolate_id'].nunique()}")
print(f"  SNP records: {len(isolate_snp_df):,}")
print(f"  SNP embedding matrix: {snp_embeddings.shape}")
print(f"  Drugs: {DRUG_COLS}")

Loaded SNP table: 12279 isolates, 5721 unique SNPs
Using first-line drugs: ['RIF_BINARY_PHENOTYPE']
SNP embeddings: (893, 64) (n_snps x embed_dim)

Prepared inputs:
  Isolates: 12279
  SNP records: 167,917
  SNP embedding matrix: (893, 64)
  Drugs: ['RIF_BINARY_PHENOTYPE']


In [7]:
# Step 2: KEGG-aware graph construction + model definition
kg = load_kegg_knowledge_graph("./kegg_data/tb_drug_knowledge_graph.json")

data = build_pathway_aware_hetero_graph(
    isolate_snp_df=isolate_snp_df,
    phenotype_df=phenotype_df,
    snp_embeddings=snp_embeddings,
    snp_list=snp_list,
    drug_cols=DRUG_COLS,
    kg=kg,
    )
data = data.to(DEVICE)

model = build_pathway_model(
    data=data,
    out_channels=len(DRUG_COLS),
    hidden_channels=64,
    dropout=0.3,
    device=DEVICE,
    )

print("\nModel ready:")
print(model.__class__.__name__)
print(f"Node types: {data.node_types}")
print(f"Edge types: {data.edge_types}")

Loaded KEGG KG from ./kegg_data/tb_drug_knowledge_graph.json
Pathway IDs available: 23
Gene->Pathway links: 36
Drug->Pathway links: 15
Graph built: 12279 isolates | 23 gene types | 893 SNP nodes
Pathway graph: 23 pathways; gene-pathway edges = 36; isolate-pathway edges = 153119

Model ready:
PathwayHGAT_AMR
Node types: ['isolate', 'ahpC', 'eis', 'embA', 'embB', 'embC', 'fabG1', 'gidB', 'gyrA', 'gyrB', 'inhA', 'iniA', 'iniB', 'iniC', 'katG', 'manB', 'ndh', 'pncA', 'rmlD', 'rpoB', 'rpoC', 'rrl', 'rrs', 'tlyA', 'pathway', 'gene']
Edge types: [('isolate', 'has_ahpC', 'ahpC'), ('ahpC', 'in_ahpC', 'isolate'), ('isolate', 'has_eis', 'eis'), ('eis', 'in_eis', 'isolate'), ('isolate', 'has_embA', 'embA'), ('embA', 'in_embA', 'isolate'), ('isolate', 'has_embB', 'embB'), ('embB', 'in_embB', 'isolate'), ('isolate', 'has_embC', 'embC'), ('embC', 'in_embC', 'isolate'), ('isolate', 'has_fabG1', 'fabG1'), ('fabG1', 'in_fabG1', 'isolate'), ('isolate', 'has_gidB', 'gidB'), ('gidB', 'in_gidB', 'isolate'),

In [8]:
# Step 3: Model training + testing + metrics
TRAIN_CFG = {
    "n_splits": 5,
    "n_epochs": 30,
    "lr": 2e-3,
    "weight_decay": 1e-4,
    "label_mask_ratio": 0.15,
    "seed": 42,
    }

cv_metrics = train_pathway_model(
    data=data,
    model=model,
    drug_cols=DRUG_COLS,
    n_splits=TRAIN_CFG["n_splits"],
    n_epochs=TRAIN_CFG["n_epochs"],
    lr=TRAIN_CFG["lr"],
    weight_decay=TRAIN_CFG["weight_decay"],
    label_mask_ratio=TRAIN_CFG["label_mask_ratio"],
    seed=TRAIN_CFG["seed"],
    )

print("\nCross-validation test metrics (mean +/- std):")
for drug, metric_map in cv_metrics.items():
    print(f"\n{drug}")
    for metric_name, (mean_val, std_val) in metric_map.items():
        print(f"  {metric_name:12s}: {mean_val:.4f} +/- {std_val:.4f}")


Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5

Cross-validation test metrics (mean +/- std):

RIF_BINARY_PHENOTYPE
  AUROC       : 0.9685 +/- 0.0030
  AUPRC       : 0.9419 +/- 0.0056
  Sensitivity : 0.9332 +/- 0.0075
  Specificity : 0.9414 +/- 0.0022
  F1          : 0.9212 +/- 0.0041
  Threshold   : 0.4600 +/- 0.1356
